## Challenge 5

1. Write a query that extracts 3 animal-related questions from the database, print these questions out
2. Use the with_generate query to provide these questions to an LLM and get it to answer these questions
3. See how many of these questions the LLM got correct by printing out the correct answer from the database!


In [ ]:
import requests
import json
#
# Download the data
resp = requests.get('https://raw.githubusercontent.com/weaviate-tutorials/quickstart/main/data/jeopardy_tiny.json')
data = json.loads(resp.text)  # Load data
    
def json_print(data):
    print(json.dumps(data, indent=2))
    
json_print(data)

In [ ]:
import weaviate
import weaviate.classes as wvc
import os

client = weaviate.connect_to_local(headers={"X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"]})

In [ ]:
client.collections.delete("Question")

In [ ]:
questions = client.collections.create("Question",
    vectorizer_config=wvc.config.Configure.Vectorizer.text2vec_openai()
)

In [ ]:
questions.data.insert_many(data)

In [ ]:
response = questions.aggregate.over_all(total_count=True)
print(response.total_count)

### Q1. Write a query that extracts 3 animal-related questions from the database, print these questions out

In [ ]:
response = questions.query.near_text(
    query="animals",
    return_properties=["question", "answer", "category"]
).do()

json_print(response)

### 2. Use the with_generate query to provide these questions to an LLM and get it to answer these questions

In [ ]:
prompt = "Answer the following question: {question}"

response = questions.query.near_text(
    query="animals",
    generate=wvc.query.Generate(single_prompt=prompt),
    return_properties=["question", "answer", "category"]
).do()

json_print(response)

### 3. See how many of these questions the LLM got correct by printing out the correct answer from the database!

In [ ]:
response = questions.query.near_text(
    query="animals",
    return_properties=["question", "answer", "category"]
).do()

json_print(response)